Laboratório de teste -ajuste de maré.
valores de maré abaixo de 1m, considerar 1m como forma de não suavizar o risco. 

Estações pluviométricas de Recife: Campina do Barreto, Dois irmãos, Torrões, Imbiribeira, Recife APAC e seus sensores.

In [7]:
import requests 
token_url = 'https://sgaa.cemaden.gov.br/SGAA/rest/controle-token/tokens' 
login = {'email': 'rafaellabmoura@gmail.com', 'password': 'Rf3m25@BrXt!'} 
response = requests.post(token_url, json=login) 
content = response.json() 
token = content['token']

In [8]:
import pandas as pd
import requests
import unicodedata

# Busca os tipos de sensores das estacoes pluviometricas de Recife solicitadas
CODIBGE_RECIFE = "2611606"
REDE = "11"
ALVOS = {
    "Campina do Barreto": ["campina do barreto"],
    "Dois Irmaos": ["dois irmaos"],
    "Torroes": ["torroes", "torrao", "torreao"],
    "Imbiribeira": ["imbiribeira"],
    "Recife APAC": ["recife apac", "apac"],
}


def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return texto


token = globals().get("token")
if not token and "content" in globals():
    token = content.get("token")

if not token:
    print("Nao foi possivel obter token.")
else:
    # 1) Buscar cadastro de estacoes de Recife
    url_estacoes = "https://sws.cemaden.gov.br/PED/rest/pcds-cadastro/dados-cadastrais"
    params_estacoes = {
        "codibge": CODIBGE_RECIFE,
        "rede": REDE,
        "formato": "JSON",
    }

    r_est = requests.get(url_estacoes, params=params_estacoes, headers={"token": token}, timeout=30)

    if r_est.status_code != 200:
        print(f"Erro ao buscar estacoes: {r_est.status_code}")
        print(r_est.text)
    else:
        df_estacoes = pd.DataFrame(r_est.json())

        if df_estacoes.empty:
            print("Nenhuma estacao encontrada para Recife.")
        else:
            # Mantem apenas estacoes pluviometricas
            df_estacoes["tipoestacao_descricao_norm"] = df_estacoes["tipoestacao_descricao"].map(normalizar_texto)
            df_pluv = df_estacoes[df_estacoes["tipoestacao_descricao_norm"].str.contains("pluviometrica", na=False)].copy()
            df_pluv["nome_norm"] = df_pluv["nome"].map(normalizar_texto)

            # 2) Garantir tabela de sensores por tipo de estacao
            if "df_sensores" in globals() and not df_sensores.empty:
                df_sensores_base = df_sensores.copy()
            else:
                url_sensores = "https://sws.cemaden.gov.br/PED/rest/pcds-tipo-estacao/sensores"
                r_sens = requests.get(url_sensores, headers={"token": token}, timeout=30)

                if r_sens.status_code != 200:
                    print(f"Erro ao buscar catalogo de sensores: {r_sens.status_code}")
                    print(r_sens.text)
                    df_sensores_base = pd.DataFrame()
                else:
                    dados_sensores_json = r_sens.json()
                    df_sensores_base = pd.json_normalize(
                        dados_sensores_json,
                        record_path="sensor",
                        meta=["tipoestacao", "tipoestacaodescricao"],
                    )

            if df_sensores_base.empty:
                print("Nao foi possivel montar o catalogo de sensores.")
            else:
                df_sensores_base["tipoestacao"] = pd.to_numeric(df_sensores_base["tipoestacao"], errors="coerce")

                resultados = []
                nao_encontradas = []

                for nome_alvo, padroes in ALVOS.items():
                    filtro_alvo = df_pluv["nome_norm"].apply(lambda nome: any(p in nome for p in padroes))
                    candidatos = df_pluv[filtro_alvo].copy()

                    if candidatos.empty:
                        nao_encontradas.append(nome_alvo)
                        continue

                    for _, est in candidatos.iterrows():
                        sensores = df_sensores_base[
                            df_sensores_base["tipoestacao"] == pd.to_numeric(est.get("id_tipoestacao"), errors="coerce")
                        ][["sensor", "sensordescricao"]].drop_duplicates()

                        if sensores.empty:
                            resultados.append(
                                {
                                    "Estacao": est.get("nome", ""),
                                    "Codigo": est.get("codestacao", ""),
                                    "Tipo_sensores": "Sem sensores mapeados",
                                }
                            )
                        else:
                            lista = sensores.sort_values(by=["sensor", "sensordescricao"]).apply(
                                lambda row: f"{int(row['sensor'])} - {row['sensordescricao']}", axis=1
                            )
                            resultados.append(
                                {
                                    "Estacao": est.get("nome", ""),
                                    "Codigo": est.get("codestacao", ""),
                                    "Tipo_sensores": " | ".join(lista.tolist()),
                                }
                            )

                if not resultados:
                    print("Nenhuma estacao alvo foi encontrada entre as pluviometricas de Recife.")
                else:
                    df_resultado = pd.DataFrame(resultados).drop_duplicates(subset=["Codigo", "Tipo_sensores"]).sort_values("Estacao")

                    print("Tipos de sensores presentes nas estacoes pluviometricas solicitadas (Recife):")
                    display(df_resultado)

                if nao_encontradas:
                    print("\nAlvos nao encontrados exatamente no cadastro:")
                    for nome in nao_encontradas:
                        print(f"- {nome}")

Tipos de sensores presentes nas estacoes pluviometricas solicitadas (Recife):


,Estacao,Codigo,Tipo_sensores
0,Campina do Barreto,261160614A,10 - Chuva | 240 - Intensidade da Precipitação
1,Dois Irmãos,261160603A,10 - Chuva | 240 - Intensidade da Precipitação
3,Imbiribeira,261160609A,10 - Chuva | 240 - Intensidade da Precipitação
4,RECIFE - APAC,261160623A,10 - Chuva | 240 - Intensidade da Precipitação
2,Torreão,261160618A,10 - Chuva | 240 - Intensidade da Precipitação


In [9]:
# Estacoes de Recife com sensor de umidade do solo
# No catalogo do CEMADEN, sensores de umidade do solo estao no tipo Geotecnica

if 'df_estacoes' not in globals() or df_estacoes.empty:
    token = token_manager.get_token()
    url_estacoes = "https://sws.cemaden.gov.br/PED/rest/pcds-cadastro/dados-cadastrais"
    params_estacoes = {"codibge": "2611606", "rede": "11", "formato": "JSON"}
    r_est = requests.get(url_estacoes, params=params_estacoes, headers={"token": token}, timeout=30)
    df_estacoes = pd.DataFrame(r_est.json()) if r_est.status_code == 200 else pd.DataFrame()

if df_estacoes.empty:
    print("Nao foi possivel carregar estacoes de Recife.")
else:
    df_umidade = df_estacoes[
        df_estacoes['tipoestacao_descricao'].fillna('').str.contains('Geotecnica|Geotécnica', case=False, regex=True)
    ][['codestacao', 'nome', 'tipoestacao_descricao']].drop_duplicates().sort_values('nome')

    print('Estacoes de Recife com sensor de umidade do solo (Geotecnica):')
    display(df_umidade)
    print(f"Total: {len(df_umidade)} estacoes")

Estacoes de Recife com sensor de umidade do solo (Geotecnica):


,codestacao,nome,tipoestacao_descricao
21,261160604G,Barreira,Geotécnica
22,261160603G,Brega e Chique,Geotécnica
6,261160616G,COMPAZ - Alto Sta. Terezinha,Geotécnica
7,261160615G,COMPESA - Alto da Brasileira,Geotécnica
10,261160613G,COMPESA - Alto da Esperança,Geotécnica
18,261160606G,Lagoa Encantada - COHAB,Geotécnica
13,261160611G,UR12 - COHAB II,Geotécnica


Total: 7 estacoes


In [10]:
import pandas as pd

if 'df_estacoes' in globals() and not df_estacoes.empty:
    df_geo = df_estacoes[
        df_estacoes['tipoestacao_descricao'].fillna('').astype(str).str.contains('Geotecnica|Geotécnica', case=False, regex=True)
    ].copy()

    print('Estacoes geotecnicas:')
    display(df_geo[['codestacao', 'nome', 'tipoestacao_descricao']].drop_duplicates().sort_values('nome'))

    if 'df_sensores_base' in globals() and not df_sensores_base.empty and 'id_tipoestacao' in df_geo.columns:
        tipos_geo = pd.to_numeric(df_geo['id_tipoestacao'], errors='coerce').dropna().unique().tolist()
        df_sens_geo = df_sensores_base[df_sensores_base['tipoestacao'].isin(tipos_geo)].copy()
        sensores_unicos = sorted(df_sens_geo['sensordescricao'].dropna().astype(str).unique().tolist()) if 'sensordescricao' in df_sens_geo.columns else []
        print('\nSensores encontrados nas estacoes geotecnicas:')
        print(sensores_unicos)
    else:
        print('\nNao foi possivel cruzar com o catalogo de sensores.')
else:
    print('df_estacoes nao esta disponivel no kernel.')

Estacoes geotecnicas:


,codestacao,nome,tipoestacao_descricao
21,261160604G,Barreira,Geotécnica
22,261160603G,Brega e Chique,Geotécnica
6,261160616G,COMPAZ - Alto Sta. Terezinha,Geotécnica
7,261160615G,COMPESA - Alto da Brasileira,Geotécnica
10,261160613G,COMPESA - Alto da Esperança,Geotécnica
18,261160606G,Lagoa Encantada - COHAB,Geotécnica
13,261160611G,UR12 - COHAB II,Geotécnica



Sensores encontrados nas estacoes geotecnicas:
['Chuva', 'Intensidade da Precipitação', 'Precipitação Acumulada - Diária', 'Umidade do Solo Nível 1', 'Umidade do Solo Nível 1 Máxima - Diária', 'Umidade do Solo Nível 1 Mínima - Diária', 'Umidade do Solo Nível 2', 'Umidade do Solo Nível 2 Máxima - Diária', 'Umidade do Solo Nível 2 Mínima - Diária', 'Umidade do Solo Nível 3', 'Umidade do Solo Nível 3 Máxima - Diária', 'Umidade do Solo Nível 3 Mínima - Diária', 'Umidade do Solo Nível 4', 'Umidade do Solo Nível 4 Máxima - Diária', 'Umidade do Solo Nível 4 Mínima - Diária', 'Umidade do Solo Nível 5', 'Umidade do Solo Nível 5 Máxima - Diária', 'Umidade do Solo Nível 5 Mínima - Diária', 'Umidade do Solo Nível 6', 'Umidade do Solo Nível 6 Máxima - Diária', 'Umidade do Solo Nível 6 Mínima - Diária']


In [37]:
# estacoes geotecnicas em funcionamento (com dados recentes)
import pandas as pd
import requests

if "token" not in globals() and "content" in globals():
    token = content.get("token")

if "token" not in globals() or not token:
    print("Nao foi possivel obter token. Rode a celula 3.")
elif "df_estacoes" not in globals() or df_estacoes.empty:
    print("df_estacoes nao esta disponivel no kernel. Rode a celula 4.")
else:
    df_geo = df_estacoes[
        df_estacoes["tipoestacao_descricao"].fillna("").astype(str).str.contains("Geotecnica|Geotécnica", case=False, regex=True)
    ].copy()

    if df_geo.empty:
        print("Nenhuma estacao geotecnica encontrada no cadastro.")
    else:
        # Se ja houver dados recentes carregados no notebook, reutiliza (evita chamadas extras)
        if "df_dados_hoje_geo" in globals() and isinstance(df_dados_hoje_geo, pd.DataFrame) and not df_dados_hoje_geo.empty:
            df_recente = df_dados_hoje_geo.copy()
        else:
            url_dados_recentes = "https://sws.cemaden.gov.br/PED/rest/pcds/pcds-dados-recentes"
            resultados = []

            for _, est in df_geo.iterrows():
                params = {
                    "codestacao": est.get("codestacao"),
                    "codibge": str(est.get("codibge", "2611606")),
                    "uf": est.get("uf", "PE"),
                    "rede": "11",
                    "formato": "JSON",
                }
                resp = requests.get(url_dados_recentes, params=params, headers={"token": token}, timeout=30)

                if resp.status_code != 200:
                    continue

                try:
                    dados = resp.json()
                except Exception:
                    continue

                if isinstance(dados, dict) and "data" in dados:
                    dados = dados["data"]
                if isinstance(dados, dict):
                    dados = [dados]

                for item in dados:
                    item["estacao"] = est.get("nome")
                    item["codestacao"] = est.get("codestacao")
                    resultados.append(item)

            df_recente = pd.DataFrame(resultados)

        if df_recente.empty:
            print("Nenhuma estacao geotecnica em funcionamento no momento (janela recente).")
        else:
            if "datahora" in df_recente.columns:
                # CEMADEN retorna em UTC; converte para horario local de Recife (UTC-3)
                df_recente["datahora_utc"] = pd.to_datetime(df_recente["datahora"], errors="coerce", utc=True)
                df_recente["datahora_recife"] = (
                    df_recente["datahora_utc"].dt.tz_convert("America/Recife").dt.tz_localize(None)
                )
            else:
                df_recente["datahora_recife"] = pd.NaT

            resumo = (
                df_recente.groupby(["estacao", "codestacao"], dropna=False)
                .agg(
                    registros=("estacao", "size"),
                    sensores=("id_sensor", lambda s: pd.Series(s).nunique()),
                    inicio=("datahora_recife", "min"),
                    fim=("datahora_recife", "max"),
                )
                .reset_index()
                .sort_values(["registros", "sensores"], ascending=False)
            )

            print("ESTACOES GEOTECNICAS EM FUNCIONAMENTO (dados recentes, hora local Recife):")
            print(f"Total: {len(resumo)} estacao(oes)")
            display(resumo)

            nomes_ativas = set(resumo["estacao"].dropna().astype(str))
            df_inativas = df_geo[~df_geo["nome"].astype(str).isin(nomes_ativas)][["codestacao", "nome"]].drop_duplicates().sort_values("nome")

            print("\nESTACOES SEM DADOS RECENTES:")
            print(f"Total: {len(df_inativas)} estacao(oes)")
            display(df_inativas)

ESTACOES GEOTECNICAS EM FUNCIONAMENTO (dados recentes, hora local Recife):
Total: 3 estacao(oes)


,estacao,codestacao,registros,sensores,inicio,fim
0,Barreira,261160604G,114,8,2026-05-04 12:40:00,2026-05-04 15:30:00
2,COMPAZ - Alto Sta. Terezinha,261160616G,114,8,2026-05-04 12:40:00,2026-05-04 15:30:00
1,Brega e Chique,261160603G,21,7,2026-05-04 13:00:00,2026-05-04 15:10:00



ESTACOES SEM DADOS RECENTES:
Total: 4 estacao(oes)


,codestacao,nome
7,261160615G,COMPESA - Alto da Brasileira
10,261160613G,COMPESA - Alto da Esperança
18,261160606G,Lagoa Encantada - COHAB
13,261160611G,UR12 - COHAB II


In [39]:
#Dados completos de Maio/2026 das estacoes geotecnicas listadas
import pandas as pd
import requests
from io import StringIO

TOKEN_URL = "https://sgaa.cemaden.gov.br/SGAA/rest/controle-token/tokens"
URL_CSV = "https://sws.cemaden.gov.br/PED/rest/pcds/dados_pcd"


def renovar_token_local():
    if "login" not in globals() or not isinstance(login, dict):
        return None, "Credenciais 'login' indisponiveis. Rode a celula 3."
    r = requests.post(TOKEN_URL, json=login, timeout=30)
    if r.status_code != 200:
        return None, f"Falha ao renovar token: HTTP {r.status_code}"
    novo = r.json().get("token")
    if not novo:
        return None, "Resposta sem token."
    globals()["token"] = novo
    if "content" in globals() and isinstance(content, dict):
        content["token"] = novo
    return novo, None


if "token" not in globals() or not token:
    token, err_tk = renovar_token_local()
    if err_tk:
        print(err_tk)

if "token" not in globals() or not token:
    print("Nao foi possivel obter token.")
else:
    # Usa o status mensal calculado na celula anterior; se nao existir, cai para todas geotecnicas
    if "df_status_maio_2026" in globals() and isinstance(df_status_maio_2026, pd.DataFrame) and not df_status_maio_2026.empty:
        estacoes_alvo = df_status_maio_2026[df_status_maio_2026["status"] == "ONLINE"][ ["codestacao", "estacao"] ].drop_duplicates().copy()
    elif "df_geo" in globals() and isinstance(df_geo, pd.DataFrame) and not df_geo.empty:
        estacoes_alvo = df_geo[["codestacao", "nome"]].drop_duplicates().rename(columns={"nome": "estacao"}).copy()
    else:
        estacoes_alvo = pd.DataFrame(columns=["codestacao", "estacao"])

    if estacoes_alvo.empty:
        print("Nenhuma estacao alvo disponivel para consulta.")
    else:
        inicio = "202605010000"
        fim = pd.Timestamp.now(tz="America/Recife").strftime("%Y%m%d%H%M")

        blocos = []
        falhas = []

        for _, est in estacoes_alvo.iterrows():
            params = {
                "codigo": est["codestacao"],
                "inicio": inicio,
                "fim": fim,
                "rede": "11",
            }
            resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)

            if resp.status_code == 401:
                token, err_tk = renovar_token_local()
                if not err_tk:
                    resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)

            if not (resp.status_code == 200 and "cod.estacao" in resp.text):
                falhas.append({"codestacao": est["codestacao"], "estacao": est["estacao"], "status_http": resp.status_code})
                continue

            linhas = [ln for ln in resp.text.splitlines() if ln.strip()]
            if linhas and linhas[0].startswith("OBS."):
                linhas = linhas[1:]

            if len(linhas) <= 1:
                continue

            df_tmp = pd.read_csv(StringIO("\n".join(linhas)), sep=";")
            if df_tmp.empty:
                continue

            # Converte UTC para hora local Recife
            if "datahora" in df_tmp.columns:
                dt_utc = pd.to_datetime(df_tmp["datahora"], errors="coerce", utc=True)
                df_tmp["datahora_recife"] = dt_utc.dt.tz_convert("America/Recife").dt.tz_localize(None)

            # Mantem apenas maio em horario local
            if "datahora_recife" in df_tmp.columns:
                ini_local = pd.Timestamp("2026-05-01 00:00:00")
                df_tmp = df_tmp[df_tmp["datahora_recife"] >= ini_local].copy()

            df_tmp["estacao_nome"] = est["estacao"]
            df_tmp["codestacao_ref"] = est["codestacao"]
            blocos.append(df_tmp)

        if not blocos:
            print("Nenhum dado de maio retornado para as estacoes selecionadas.")
            if falhas:
                display(pd.DataFrame(falhas))
        else:
            df_maio_estacoes = pd.concat(blocos, ignore_index=True)

            # Ordenacao e colunas principais
            cols_inicio = [
                "codestacao_ref", "estacao_nome", "datahora_recife", "datahora",
                "sensor", "valor", "qualificacao", "municipio", "uf"
            ]
            cols_inicio = [c for c in cols_inicio if c in df_maio_estacoes.columns]
            df_maio_estacoes = df_maio_estacoes[cols_inicio + [c for c in df_maio_estacoes.columns if c not in cols_inicio]]

            if "datahora_recife" in df_maio_estacoes.columns:
                df_maio_estacoes = df_maio_estacoes.sort_values(["estacao_nome", "datahora_recife", "sensor"], na_position="last")

            print("DADOS DE MAIO/2026 - ESTACOES GEOTECNICAS")
            print(f"Estacoes com dados: {df_maio_estacoes['estacao_nome'].nunique()}")
            print(f"Total de registros: {len(df_maio_estacoes)}")

            resumo = (
                df_maio_estacoes.groupby(["estacao_nome", "codestacao_ref"], dropna=False)
                .agg(
                    registros=("estacao_nome", "size"),
                    sensores=("sensor", lambda s: pd.Series(s).nunique()),
                    inicio_recife=("datahora_recife", "min"),
                    fim_recife=("datahora_recife", "max"),
                )
                .reset_index()
                .sort_values("registros", ascending=False)
            )
            display(resumo)
            display(df_maio_estacoes.head(100))

            nome_saida = "dados_maio_2026_estacoes_geotecnicas.csv"
            df_maio_estacoes.to_csv(nome_saida, index=False, encoding="utf-8-sig")
            print(f"Arquivo salvo: {nome_saida}")

            if falhas:
                print("\nFalhas de consulta:")
                display(pd.DataFrame(falhas))

DADOS DE MAIO/2026 - ESTACOES GEOTECNICAS
Estacoes com dados: 3
Total de registros: 8445


,estacao_nome,codestacao_ref,registros,sensores,inicio_recife,fim_recife
0,Barreira,261160604G,4192,21,2026-05-01,2026-05-05 12:30:00
2,COMPAZ - Alto Sta. Terezinha,261160616G,3446,21,2026-05-01,2026-05-05 12:30:00
1,Brega e Chique,261160603G,807,18,2026-05-01,2026-05-05 12:10:00


,codestacao_ref,estacao_nome,datahora_recife,datahora,sensor,valor,qualificacao,municipio,uf,cod.estacao,nome,latitude,longitude,offset
3,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel1,14.917540,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
0,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel2,6.552492,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
4,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel3,3.072646,2,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
1,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel4,11.989940,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
5,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel5,1.471403,2,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,umidade_solo_nivel4,11.992740,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
90,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,umidade_solo_nivel5,1.471335,2,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
93,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,umidade_solo_nivel6,36.601010,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
98,261160604G,Barreira,2026-05-01 02:30:00,2026-05-01 05:30:00,umidade_solo_nivel1,14.937670,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN


Arquivo salvo: dados_maio_2026_estacoes_geotecnicas.csv


In [12]:
#dados geotecnicos -umidade do solo - mes de maio completo

# Dados completos e isolados de Maio/2026 das estacoes geotecnicas de Recife
import pandas as pd
import requests
from io import StringIO

# Configurações de endpoints institucionais do Cemaden
TOKEN_URL = "https://sgaa.cemaden.gov.br/SGAA/rest/controle-token/tokens"
URL_CSV = "https://sws.cemaden.gov.br/PED/rest/pcds/dados_pcd"

# Suas credenciais de acesso privado fornecidas
login = {'email': 'rafaellabmoura@gmail.com', 'password': 'Rf3m25@BrXt!'}

def renovar_token_local():
    if "login" not in globals() or not isinstance(login, dict):
        return None, "Credenciais 'login' indisponiveis na memoria global."
    
    print("🔄 Solicitando/Renovando token junto ao SGAA-Cemaden...")
    try:
        r = requests.post(TOKEN_URL, json=login, timeout=30)
        if r.status_code != 200:
            return None, f"Falha ao renovar token: HTTP {r.status_code}"
        novo = r.json().get("token")
        if not novo:
            return None, "Resposta do servidor sem a chave 'token'."
        
        globals()["token"] = novo
        if "content" in globals() and isinstance(content, dict):
            content["token"] = novo
        return novo, None
    except Exception as e:
        return None, f"Erro de conexao ao autenticar: {e}"

# Inicializa ou renova o token para a sessão atual
if "token" not in globals() or not token:
    token, err_tk = renovar_token_local()
    if err_tk:
        print(err_tk)

if "token" not in globals() or not token:
    print("❌ Nao foi possivel obter o token de acesso. Verifique a rede ou as credenciais.")
else:
    # ---------------------------------------------------------------------
    # DEFINIÇÃO IN CONDICIONAL DAS SUAS 7 ESTAÇÕES GEOTÉCNICAS ALVO
    # ---------------------------------------------------------------------
    dados_pcds_recife = {
        "codestacao": [
            "261160604G",  # Barreira
            "261160603G",  # Brega e Chique
            "261160616G",  # COMPAZ - Alto Sta. Terezinha
            "261160615G",  # COMPESA - Alto da Brasileira
            "261160613G",  # COMPESA - Alto da Esperança
            "261160606G",  # Lagoa Encantada - COHAB
            "261160611G"   # UR12 - COHAB II
        ],
        "estacao": [
            "Barreira", 
            "Brega e Chique", 
            "COMPAZ - Alto Sta. Terezinha", 
            "COMPESA - Alto da Brasileira", 
            "COMPESA - Alto da Esperança", 
            "Lagoa Encantada - COHAB", 
            "UR12 - COHAB II"
        ]
    }
    estacoes_alvo = pd.DataFrame(dados_pcds_recife)

    # ---------------------------------------------------------------------
    # CONFIGURAÇÃO DE MAIO COMPLETO PARA A EXTRAÇÃO (Padrão API YYYYMMDDHHMM)
    # ---------------------------------------------------------------------
    inicio = "202605010000"
    fim = "202605312359"  # Trancado no último minuto de maio

    blocos = []
    falhas = []

    print(f"🔎 Conectado. Iniciando varredura para as {len(estacoes_alvo)} PCDs Geotécnicas...")

    for _, est in estacoes_alvo.iterrows():
        params = {
            "codigo": str(est["codestacao"]).strip(),
            "inicio": inicio,
            "fim": fim,
            "rede": "11",
        }
        
        try:
            resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)
        except Exception as e:
            falhas.append({"codestacao": est["codestacao"], "estacao": est["estacao"], "status_http": f"Erro de Conexão: {e}"})
            continue

        # Se o token expirou no meio da requisição de lote, renova automaticamente
        if resp.status_code == 401:
            token, err_tk = renovar_token_local()
            if not err_tk:
                resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)

        # Valida se o retorno é um CSV válido do Cemaden
        if not (resp.status_code == 200 and ("cod.estacao" in resp.text or "codestacao" in resp.text.lower())):
            falhas.append({"codestacao": est["codestacao"], "estacao": est["estacao"], "status_http": resp.status_code})
            continue

        linhas = [ln for ln in resp.text.splitlines() if ln.strip()]
        if linhas and linhas[0].startswith("OBS."):
            linhas = linhas[1:]

        if len(linhas) <= 1:
            continue

        df_tmp = pd.read_csv(StringIO("\n".join(linhas)), sep=";")
        if df_tmp.empty:
            continue

        # Padroniza as colunas nativas vindas da API em minúsculo para evitar conflitos
        df_tmp.columns = df_tmp.columns.str.lower()

        # Converte a estampa de tempo UTC padrão para o Horário Local de Recife
        col_data = "datahora" if "datahora" in df_tmp.columns else "data_hora" if "data_hora" in df_tmp.columns else None
        if col_data:
            dt_utc = pd.to_datetime(df_tmp[col_data], errors="coerce", utc=True)
            df_tmp["datahora_recife"] = dt_utc.dt.tz_convert("America/Recife").dt.tz_localize(None)

        # FILTRO DE SEGURANÇA: Isola o dataframe entre 01/05 e 31/05 na hora local
        if "datahora_recife" in df_tmp.columns:
            ini_local = pd.Timestamp("2026-05-01 00:00:00")
            fim_local = pd.Timestamp("2026-05-31 23:59:59")
            df_tmp = df_tmp[(df_tmp["datahora_recife"] >= ini_local) & (df_tmp["datahora_recife"] <= fim_local)].copy()

        # Adiciona os metadados de identificação amigáveis
        df_tmp["estacao_nome"] = est["estacao"]
        df_tmp["codestacao_ref"] = est["codestacao"]
        blocos.append(df_tmp)

    # ---------------------------------------------------------------------
    # CONSOLIDAÇÃO E EXIBIÇÃO COMPLETA DAS COLUNAS (INCLUINDO COORDENADAS)
    # ---------------------------------------------------------------------
    if not blocos:
        print("\n❌ Nenhum dado foi retornado para as estações selecionadas no período informado.")
        if falhas:
            print("\nTabela de monitoramento de falhas na API:")
            display(pd.DataFrame(falhas))
    else:
        df_maio_estacoes = pd.concat(blocos, ignore_index=True)

        # Ajusta possíveis variações de nomenclatura do caractere de ponto no código da estação
        if "cod.estacao" in df_maio_estacoes.columns:
            df_maio_estacoes = df_maio_estacoes.rename(columns={"cod.estacao": "codestacao"})

        # Reordena posicionando chaves de tempo e geolocalização no início do DataFrame para facilitar a leitura
        cols_organizadas = [
            "codestacao_ref", "estacao_nome", "datahora_recife", "datahora",
            "latitude", "longitude", "sensor", "valor", "qualificacao", "municipio", "uf"
        ]
        
        # Mantém as colunas solicitadas que existirem no retorno + todas as outras remanescentes
        cols_existentes = [c for c in cols_organizadas if c in df_maio_estacoes.columns]
        outras_cols = [c for c in df_maio_estacoes.columns if c not in cols_existentes]
        df_maio_estacoes = df_maio_estacoes[cols_existentes + outras_cols]

        # Ordenação cronológica e por tipo de sensor
        if "datahora_recife" in df_maio_estacoes.columns:
            df_maio_estacoes = df_maio_estacoes.sort_values(["estacao_nome", "datahora_recife", "sensor"], na_position="last")

        print("\n=======================================================")
        print("🎉 SUCESSO! BASE DE MAIO/2026 CONSOLIDADA")
        print(f"   Estações ativas identificadas: {df_maio_estacoes['estacao_nome'].nunique()}")
        print(f"   Total de registros de umidade/sensores extraídos: {len(df_maio_estacoes)}")
        print("=======================================================\n")

        # Exibe o sumário consolidado das séries capturadas
        resumo = (
            df_maio_estacoes.groupby(["estacao_nome", "codestacao_ref"], dropna=False)
            .agg(
                registros=("estacao_nome", "size"),
                sensores=("sensor", lambda s: pd.Series(s).nunique()),
                inicio_recife=("datahora_recife", "min"),
                fim_recife=("datahora_recife", "max"),
            )
            .reset_index()
            .sort_values("registros", ascending=False)
        )
        
        print("📋 RESUMO DA DISPONIBILIDADE POR ESTAÇÃO:")
        display(resumo)
        
        print("\n👀 MONITORAMENTO DE COLUNAS DISPONÍVEIS (Primeiras 100 linhas):")
        display(df_maio_estacoes.head(100))

        # Exportação final em UTF-8 estruturado com separador de vírgula padrão
        nome_saida = "dados_maio_2026_estacoes_geotecnicas.csv"
        df_maio_estacoes.to_csv(nome_saida, index=False, encoding="utf-8-sig")
        print(f"\n💾 Arquivo indexado e salvo com sucesso no diretório: '{nome_saida}'")

        if falhas:
            print("\n⚠️ Alerta: As seguintes estações apresentaram falhas de comunicação ou restrição de dados:")
            display(pd.DataFrame(falhas))

🔎 Conectado. Iniciando varredura para as 7 PCDs Geotécnicas...

🎉 SUCESSO! BASE DE MAIO/2026 CONSOLIDADA
   Estações ativas identificadas: 3
   Total de registros de umidade/sensores extraídos: 54372

📋 RESUMO DA DISPONIBILIDADE POR ESTAÇÃO:


,estacao_nome,codestacao_ref,registros,sensores,inicio_recife,fim_recife
0,Barreira,261160604G,24778,21,2026-05-01,2026-05-31 20:50:00
2,COMPAZ - Alto Sta. Terezinha,261160616G,24082,21,2026-05-01,2026-05-31 20:50:00
1,Brega e Chique,261160603G,5512,18,2026-05-01,2026-05-31 20:10:00



👀 MONITORAMENTO DE COLUNAS DISPONÍVEIS (Primeiras 100 linhas):


,codestacao_ref,estacao_nome,datahora_recife,datahora,latitude,longitude,sensor,valor,qualificacao,municipio,uf,codestacao,nome,offset
3,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,-8.024215,-34.964622,umidade_solo_nivel1,14.917540,4,RECIFE,PE,261160604G,Barreira,NaN
1,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,-8.024215,-34.964622,umidade_solo_nivel2,6.552492,4,RECIFE,PE,261160604G,Barreira,NaN
4,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,-8.024215,-34.964622,umidade_solo_nivel3,3.072646,2,RECIFE,PE,261160604G,Barreira,NaN
0,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,-8.024215,-34.964622,umidade_solo_nivel4,11.989940,4,RECIFE,PE,261160604G,Barreira,NaN
5,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,-8.024215,-34.964622,umidade_solo_nivel5,1.471403,2,RECIFE,PE,261160604G,Barreira,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,-8.024215,-34.964622,umidade_solo_nivel4,11.992740,4,RECIFE,PE,261160604G,Barreira,NaN
91,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,-8.024215,-34.964622,umidade_solo_nivel5,1.471335,2,RECIFE,PE,261160604G,Barreira,NaN
93,261160604G,Barreira,2026-05-01 02:20:00,2026-05-01 05:20:00,-8.024215,-34.964622,umidade_solo_nivel6,36.601010,4,RECIFE,PE,261160604G,Barreira,NaN
98,261160604G,Barreira,2026-05-01 02:30:00,2026-05-01 05:30:00,-8.024215,-34.964622,umidade_solo_nivel1,14.937670,4,RECIFE,PE,261160604G,Barreira,NaN



💾 Arquivo indexado e salvo com sucesso no diretório: 'dados_maio_2026_estacoes_geotecnicas.csv'

⚠️ Alerta: As seguintes estações apresentaram falhas de comunicação ou restrição de dados:


,codestacao,estacao,status_http
0,261160615G,COMPESA - Alto da Brasileira,202
1,261160613G,COMPESA - Alto da Esperança,202
2,261160606G,Lagoa Encantada - COHAB,202
3,261160611G,UR12 - COHAB II,202


In [13]:
#as coordenadas

# Isolar a geolocalização exata de cada estação a partir dos dados obtidos
import pandas as pd

# 1. Carrega o arquivo gerado pelo script anterior
try:
    df_maio = pd.read_csv("dados_maio_2026_estacoes_geotecnicas.csv")
    
    # 2. Agrupa pelas colunas de identificação e coordenadas para pegar os valores únicos
    # Usamos o drop_duplicates para limpar os milhares de registros horários e focar no cadastro
    df_coordenadas = df_maio[[
        "codestacao_ref", 
        "estacao_nome", 
        "latitude", 
        "longitude"
    ]].drop_duplicates().reset_index(drop=True)
    
    # 3. Exibe a tabela final de geolocalização para o teu Artigo/TCC
    print("=======================================================")
    print("📍 COORDENADAS GEOGRÁFICAS DAS ESTAÇÕES (RECIFE)")
    print("=======================================================")
    display(df_coordenadas)
    
    # 4. Opcional: Salva esta tabela de coordenadas num arquivo separado para fazer mapas
    df_coordenadas.to_csv("coordenadas_estacoes_geotecnicas.csv", index=False, encoding="utf-8-sig")
    print("\n💾 Tabela de coordenadas salva em: 'coordenadas_estacoes_geotecnicas.csv'")

except FileNotFoundError:
    print("⚠️ O arquivo 'dados_maio_2026_estacoes_geotecnicas.csv' não foi encontrado.")
    print("Certifica-te de rodar esta célula na mesma pasta onde o código anterior salvou os dados.")

📍 COORDENADAS GEOGRÁFICAS DAS ESTAÇÕES (RECIFE)


,codestacao_ref,estacao_nome,latitude,longitude
0,261160604G,Barreira,-8.024215,-34.964622
1,261160603G,Brega e Chique,-8.038048,-34.979298
2,261160616G,COMPAZ - Alto Sta. Terezinha,-8.009190,-34.902819



💾 Tabela de coordenadas salva em: 'coordenadas_estacoes_geotecnicas.csv'


In [15]:
#dados completos de Maio/2026 pela API Open-Meteo

# Buscar dados de umidade do Open-Meteo em tempo real para os mesmos pontos válidos
import pandas as pd
import requests

# Lista com as 3 estações que funcionaram no seu teste do Cemaden
estacoes_validas = [
    {"nome": "Barreira", "lat": -8.024215, "lon": -34.964622},
    {"nome": "Brega e Chique", "lat": -8.038048, "lon": -34.979298},
    {"nome": "COMPAZ - Alto Sta. Terezinha", "lat": -8.009190, "lon": -34.902819}
]

# Período de Maio completo
DATA_INICIO = "2026-05-01"
DATA_FIM = "2026-05-31"

blocos_openmeteo = []

print("⏳ Buscando umidade do solo no Open-Meteo para os pontos de Recife...")

for est in estacoes_validas:
    # Alterado para o endpoint 'forecast' combinado com past_days para evitar o delay do ERA5
    url = "https://api.open-meteo.com/v1/forecast"
    
    params = {
        "latitude": est["lat"],
        "longitude": est["lon"],
        "start_date": DATA_INICIO,
        "end_date": DATA_FIM,
        "hourly": ["soil_moisture_0_to_1cm", "soil_moisture_3_to_9cm", "soil_moisture_27_to_81cm"],
        "timezone": "America/Recife"
    }
    
    try:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            dados_api = resp.json()
            
            if "hourly" in dados_api:
                df_hora = pd.DataFrame({
                    "datahora_recife": pd.to_datetime(dados_api["hourly"]["time"]),
                    "estacao_nome": est["nome"],
                    "latitude_om": est["lat"],
                    "longitude_om": est["lon"],
                    "openmeteo_0_1cm": dados_api["hourly"]["soil_moisture_0_to_1cm"],
                    "openmeteo_3_9cm": dados_api["hourly"]["soil_moisture_3_to_9cm"],
                    "openmeteo_27_81cm": dados_api["hourly"]["soil_moisture_27_to_81cm"]
                })
                blocos_openmeteo.append(df_hora)
                print(f"✅ Dados obtidos com sucesso para: {est['nome']}")
            else:
                print(f"⚠️ API respondeu, mas não encontrou variáveis horárias para {est['nome']}")
        else:
            print(f"❌ Erro na API para {est['nome']}: HTTP {resp.status_code}")
            
    except Exception as e:
        print(f"❌ Falha na conexão para {est['nome']}: {e}")

# Consolidação dos resultados
if blocos_openmeteo:
    df_om_maio = pd.concat(blocos_openmeteo, ignore_index=True)
    
    # Remove possíveis fusos horários da string para não atrapalhar o merge depois
    df_om_maio["datahora_recife"] = df_om_maio["datahora_recife"].dt.tz_localize(None)
    
    df_om_maio.to_csv("openmeteo_maio_validadas.csv", index=False, encoding="utf-8-sig")
    print("\n=======================================================")
    print("🎉 SUCESSO! Tabela 'openmeteo_maio_validadas.csv' gerada.")
    print(f"   Total de linhas extraídas: {len(df_om_maio)}")
    print("=======================================================\n")
    display(df_om_maio.head(10))
else:
    print("\n❌ Nenhum dado foi retornado. Verifique a sua conexão com a internet.")

⏳ Buscando umidade do solo no Open-Meteo para os pontos de Recife...
✅ Dados obtidos com sucesso para: Barreira
✅ Dados obtidos com sucesso para: Brega e Chique
✅ Dados obtidos com sucesso para: COMPAZ - Alto Sta. Terezinha

🎉 SUCESSO! Tabela 'openmeteo_maio_validadas.csv' gerada.
   Total de linhas extraídas: 2232



,datahora_recife,estacao_nome,latitude_om,longitude_om,openmeteo_0_1cm,openmeteo_3_9cm,openmeteo_27_81cm
0,2026-05-01 00:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
1,2026-05-01 01:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
2,2026-05-01 02:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
3,2026-05-01 03:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
4,2026-05-01 04:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
5,2026-05-01 05:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
6,2026-05-01 06:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
7,2026-05-01 07:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
8,2026-05-01 08:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN
9,2026-05-01 09:00:00,Barreira,-8.024215,-34.964622,NaN,NaN,NaN


In [47]:
# Base de trabalho no pandas: dados_maio_2026_estacoes_geotecnicas.csv
import pandas as pd

arquivo_base = "dados_maio_2026_estacoes_geotecnicas.csv"

df_geotecnica = pd.read_csv(arquivo_base, encoding="utf-8-sig")



In [48]:
print("\nColunas disponiveis:")
print(df_geotecnica.columns.tolist())

print("\nResumo por estacao:")
resumo_estacao = (
    df_geotecnica.groupby("estacao_nome", dropna=False)
    .agg(
        registros=("estacao_nome", "size"),
        sensores=("sensor", lambda s: pd.Series(s).nunique()),
        inicio=("datahora_recife", "min"),
        fim=("datahora_recife", "max"),
    )
    .sort_values("registros", ascending=False)
)
display(resumo_estacao)

print("\nAmostra dos dados:")
display(df_geotecnica.head(20))


Colunas disponiveis:
['codestacao_ref', 'estacao_nome', 'datahora_recife', 'datahora', 'sensor', 'valor', 'qualificacao', 'municipio', 'uf', 'cod.estacao', 'nome', 'latitude', 'longitude', 'offset']

Resumo por estacao:


,registros,sensores,inicio,fim
estacao_nome,,,,
Barreira,4192,21,2026-05-01 00:00:00,2026-05-05 12:30:00
COMPAZ - Alto Sta. Terezinha,3446,21,2026-05-01 00:00:00,2026-05-05 12:30:00
Brega e Chique,807,18,2026-05-01 00:00:00,2026-05-05 12:10:00



Amostra dos dados:


,codestacao_ref,estacao_nome,datahora_recife,datahora,sensor,valor,qualificacao,municipio,uf,cod.estacao,nome,latitude,longitude,offset
0,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel1,14.917540,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
1,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel2,6.552492,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
2,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel3,3.072646,2,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
3,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel4,11.989940,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
4,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel5,1.471403,2,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
5,261160604G,Barreira,2026-05-01 00:00:00,2026-05-01 03:00:00,umidade_solo_nivel6,36.601800,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
6,261160604G,Barreira,2026-05-01 00:10:00,2026-05-01 03:10:00,chuva,0.000000,0,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
7,261160604G,Barreira,2026-05-01 00:10:00,2026-05-01 03:10:00,intensidade_precipitacao,0.000000,0,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
8,261160604G,Barreira,2026-05-01 00:10:00,2026-05-01 03:10:00,umidade_solo_nivel1,14.927020,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN
9,261160604G,Barreira,2026-05-01 00:10:00,2026-05-01 03:10:00,umidade_solo_nivel2,6.552216,4,RECIFE,PE,261160604G,Barreira,-8.024215,-34.964622,NaN


In [50]:
#Seleção das colunas mais relevantes para analise inicial
df_geo = df_geotecnica[["estacao_nome", "datahora_recife", "sensor", "valor", "qualificacao", "longitude", "latitude"]]

print(f"Shape de df_geo: {df_geo.shape}")
print(f"\nColunas em df_geo:\n{df_geo.columns.tolist()}")
print(f"\nAmostra:")
display(df_geo.head(10))

Shape de df_geo: (8445, 7)

Colunas em df_geo:
['estacao_nome', 'datahora_recife', 'sensor', 'valor', 'qualificacao', 'longitude', 'latitude']

Amostra:


,estacao_nome,datahora_recife,sensor,valor,qualificacao,longitude,latitude
0,Barreira,2026-05-01 00:00:00,umidade_solo_nivel1,14.917540,4,-34.964622,-8.024215
1,Barreira,2026-05-01 00:00:00,umidade_solo_nivel2,6.552492,4,-34.964622,-8.024215
2,Barreira,2026-05-01 00:00:00,umidade_solo_nivel3,3.072646,2,-34.964622,-8.024215
3,Barreira,2026-05-01 00:00:00,umidade_solo_nivel4,11.989940,4,-34.964622,-8.024215
4,Barreira,2026-05-01 00:00:00,umidade_solo_nivel5,1.471403,2,-34.964622,-8.024215
5,Barreira,2026-05-01 00:00:00,umidade_solo_nivel6,36.601800,4,-34.964622,-8.024215
6,Barreira,2026-05-01 00:10:00,chuva,0.000000,0,-34.964622,-8.024215
7,Barreira,2026-05-01 00:10:00,intensidade_precipitacao,0.000000,0,-34.964622,-8.024215
8,Barreira,2026-05-01 00:10:00,umidade_solo_nivel1,14.927020,4,-34.964622,-8.024215
9,Barreira,2026-05-01 00:10:00,umidade_solo_nivel2,6.552216,4,-34.964622,-8.024215


In [51]:
#verificar valores ausentes da coluna de valor
print("\nValores ausentes na coluna 'valor':")
print(df_geo["valor"].isna().sum())
print("\nPercentual de valores ausentes:")
print(f"{(df_geo['valor'].isna().mean() * 100):.2f}%")



Valores ausentes na coluna 'valor':
0

Percentual de valores ausentes:
0.00%


In [52]:
#verificar valores nulos ou nao numericos na coluna de valor
print("\nValores nulos ou nao numericos na coluna 'valor':")
print(df_geo["valor"].isna().sum())


Valores nulos ou nao numericos na coluna 'valor':
0


In [53]:
#valores unicos de sensor
print("\nSensores presentes nos dados:")
print(df_geo["sensor"].unique())


Sensores presentes nos dados:
['umidade_solo_nivel1' 'umidade_solo_nivel2' 'umidade_solo_nivel3'
 'umidade_solo_nivel4' 'umidade_solo_nivel5' 'umidade_solo_nivel6' 'chuva'
 'intensidade_precipitacao' 'precipitacao_acumulada'
 'umidade_solo_nivel1_max' 'umidade_solo_nivel1_min'
 'umidade_solo_nivel2_max' 'umidade_solo_nivel2_min'
 'umidade_solo_nivel3_max' 'umidade_solo_nivel3_min'
 'umidade_solo_nivel4_max' 'umidade_solo_nivel4_min'
 'umidade_solo_nivel5_max' 'umidade_solo_nivel5_min'
 'umidade_solo_nivel6_max' 'umidade_solo_nivel6_min']


In [73]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# Mapeamento de profundidades dos sensores de umidade
profundidade_map = {
    'umidade_solo_nivel1': '0.5m',
    'umidade_solo_nivel2': '1m',
    'umidade_solo_nivel3': '1.5m',
    'umidade_solo_nivel4': '2.0m',
    'umidade_solo_nivel5': '2.5m',
    'umidade_solo_nivel6': '3m',
}

# Mapeamento de cores para cada nível (cores bem distintas)
cores_map = {
    'umidade_solo_nivel1 (0.5m)': '#1f77b4',   # azul
    'umidade_solo_nivel2 (1m)': '#ff7f0e',     # laranja
    'umidade_solo_nivel3 (1.5m)': '#2ca02c',   # verde
    'umidade_solo_nivel4 (2.0m)': '#d62728',   # vermelho
    'umidade_solo_nivel5 (2.5m)': '#9467bd',   # roxo
    'umidade_solo_nivel6 (3m)': '#e377c2',     # rosa
}

def get_barreira_intervalo(df):
    tmp = df.copy()
    tmp['datahora_recife'] = pd.to_datetime(tmp['datahora_recife'], errors='coerce')
    
    # Filtra intervalo 12:00 - 13:00 e sensores de umidade apenas
    start_dt = pd.Timestamp('2026-05-01 12:00:00')
    end_dt = pd.Timestamp('2026-05-01 13:00:00')
    
    result = tmp[
        (tmp['estacao_nome'] == 'Barreira') & 
        (tmp['datahora_recife'] >= start_dt) & 
        (tmp['datahora_recife'] < end_dt) &
        (tmp['sensor'].str.contains('umidade_solo', case=False, na=False))
    ].copy()
    
    # Adiciona coluna com profundidade
    result['profundidade'] = result['sensor'].map(profundidade_map)
    result['sensor_label'] = result['sensor'] + ' (' + result['profundidade'] + ')'
    
    return result

result = pd.DataFrame()
if 'df_geo' in globals() and isinstance(df_geo, pd.DataFrame):
    result = get_barreira_intervalo(df_geo)
elif 'df_geo_wide' in globals() and isinstance(df_geo_wide, pd.DataFrame):
    result = get_barreira_intervalo(df_geo_wide)

if result.empty:
    print('Nenhum registro encontrado para sensores de Barreira entre 12h-13h.')
else:
    result = result.sort_values(['sensor', 'datahora_recife'])
    print(f"Total de pontos: {len(result)}")
    print(f"Sensores: {result['sensor_label'].unique().tolist()}")
    display(result[['datahora_recife', 'sensor', 'profundidade', 'valor', 'qualificacao']].head(20))

    fig = px.line(
        result,
        x='datahora_recife',
        y='valor',
        color='sensor_label',
        markers=True,
        title='Barreira - Umidade do Solo por Profundidade (12h-13h em 2026-05-01)',
        hover_data=['profundidade', 'qualificacao', 'valor'],
        color_discrete_map=cores_map,
    )
    fig.update_layout(
        xaxis_title='Hora (Recife)',
        yaxis_title='Umidade (%)',
        height=500,
        hovermode='x unified',
        legend_title='Profundidade',
    )
    fig.show()


Total de pontos: 36
Sensores: ['umidade_solo_nivel1 (0.5m)', 'umidade_solo_nivel2 (1m)', 'umidade_solo_nivel3 (1.5m)', 'umidade_solo_nivel4 (2.0m)', 'umidade_solo_nivel5 (2.5m)', 'umidade_solo_nivel6 (3m)']


,datahora_recife,sensor,profundidade,valor,qualificacao
546,2026-05-01 12:00:00,umidade_solo_nivel1,0.5m,21.781430,4
554,2026-05-01 12:10:00,umidade_solo_nivel1,0.5m,21.381480,4
562,2026-05-01 12:20:00,umidade_solo_nivel1,0.5m,20.339720,4
570,2026-05-01 12:30:00,umidade_solo_nivel1,0.5m,19.212420,4
578,2026-05-01 12:40:00,umidade_solo_nivel1,0.5m,18.538810,4
586,2026-05-01 12:50:00,umidade_solo_nivel1,0.5m,18.309850,4
547,2026-05-01 12:00:00,umidade_solo_nivel2,1m,9.490433,4
555,2026-05-01 12:10:00,umidade_solo_nivel2,1m,8.947117,4
563,2026-05-01 12:20:00,umidade_solo_nivel2,1m,8.512571,4
571,2026-05-01 12:30:00,umidade_solo_nivel2,1m,8.295611,4


comparação umidade estações geotécnicas cemaden maio 2026 e dados da api open-meteo

In [1]:
import pandas as pd
import requests
import json
from datetime import datetime

# =========================================================================
# PASSO 1: Configuração do Período de Estudo (Maio Completo)
# =========================================================================
DATA_INICIO = "2026-05-01"
DATA_FIM = "2026-05-31"

# =========================================================================
# PASSO 2: Carregar as Coordenadas do seu Arquivo do Cemaden
# =========================================================================
# Subescreva o nome abaixo com o nome real do seu arquivo do Cemaden
try:
    df_cemaden = pd.read_csv("seus_dados_cemaden.csv", sep=";")
    
    # Extrai as coordenadas únicas das estações contidas nas colunas do seu arquivo
    # IMPORTANTE: Ajuste os nomes 'latitude' e 'longitude' conforme estão na sua tabela
    latitudes = df_cemaden["latitude"].dropna().unique()
    longitudes = df_cemaden["longitude"].dropna().unique()
    
    print(f"📍 Detectadas {len(latitudes)} estações no arquivo do Cemaden.")
except FileNotFoundError:
    print("⚠️ Arquivo do Cemaden não encontrado no diretório. Usando coordenadas padrão de Recife para o exemplo...")
    # Coordenadas padrão caso rode como teste (Campina do Barreto, Imbiribeira, etc.)
    latitudes = [-8.0214, -8.1147] 
    longitudes = [-34.8872, -34.9042]

# =========================================================================
# PASSO 3: Requisição Automatizada na API do Open-Meteo
# =========================================================================
print(f"⏳ Iniciando extração do Open-Meteo para todo o mês de Maio...")
lista_frames = []

for lat, lon in zip(latitudes, longitudes):
    # Endpoint de Dados Históricos (ERA5-Land) do Open-Meteo
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": DATA_INICIO,
        "end_date": DATA_FIM,
        "hourly": ["soil_moisture_0_to_1cm", "soil_moisture_3_to_9cm", "soil_moisture_27_to_81cm"],
        "timezone": "America/Recife"
    }
    
    resposta = requests.get(url, params=params)
    
    if resposta.status_code == 200:
        dados_api = resposta.json()
        df_hora = pd.DataFrame({
            "Data_Hora": dados_api["hourly"]["time"],
            "Latitude": lat,
            "Longitude": lon,
            "OpenMeteo_Superficial_0_1cm": dados_api["hourly"]["soil_moisture_0_to_1cm"],
            "OpenMeteo_Intermediaria_3_9cm": dados_api["hourly"]["soil_moisture_3_to_9cm"],
            "OpenMeteo_Profunda_27_81cm": dados_api["hourly"]["soil_moisture_27_to_81cm"]
        })
        lista_frames.append(df_hora)
        print(f"✅ Dados coletados com sucesso para a coordenada: {lat}, {lon}")
    else:
        print(f"❌ Erro ao acessar a API na coordenada {lat}, {lon}: Código {resposta.status_code}")

# =========================================================================
# PASSO 4: Salvar o Histórico do Open-Meteo para a Comparação
# =========================================================================
if lista_frames:
    df_openmeteo_maio = pd.concat(lista_frames, ignore_index=True)
    df_openmeteo_maio.to_csv("openmeteo_maio_completo.csv", index=False, sep=";")
    print("\n=======================================================")
    print("🎉 SUCESSO! O arquivo 'openmeteo_maio_completo.csv' foi gerado.")
    print(f"   Total de registros horários extraídos: {len(df_openmeteo_maio)} linhas.")
    print("=======================================================")
else:
    print("⚠️ Nenhum dado foi extraído da API.")

⚠️ Arquivo do Cemaden não encontrado no diretório. Usando coordenadas padrão de Recife para o exemplo...
⏳ Iniciando extração do Open-Meteo para todo o mês de Maio...
✅ Dados coletados com sucesso para a coordenada: -8.0214, -34.8872
✅ Dados coletados com sucesso para a coordenada: -8.1147, -34.9042

🎉 SUCESSO! O arquivo 'openmeteo_maio_completo.csv' foi gerado.
   Total de registros horários extraídos: 1488 linhas.


In [4]:
# Dados completos e isolados de Maio/2026 das estacoes geotecnicas listadas
import pandas as pd
import requests
from io import StringIO

TOKEN_URL = "https://sgaa.cemaden.gov.br/SGAA/rest/controle-token/tokens"
URL_CSV = "https://sws.cemaden.gov.br/PED/rest/pcds/dados_pcd"

def renovar_token_local():
    if "login" not in globals() or not isinstance(login, dict):
        return None, "Credenciais 'login' indisponiveis. Rode a celula de configuracao do login primeiro."
    r = requests.post(TOKEN_URL, json=login, timeout=30)
    if r.status_code != 200:
        return None, f"Falha ao renovar token: HTTP {r.status_code}"
    novo = r.json().get("token")
    if not novo:
        return None, "Resposta sem token."
    globals()["token"] = novo
    if "content" in globals() and isinstance(content, dict):
        content["token"] = novo
    return novo, None

# Tenta obter ou renovar o token
if "token" not in globals() or not token:
    token, err_tk = renovar_token_local()
    if err_tk:
        print(err_tk)

if "token" not in globals() or not token:
    print("Nao foi possivel obter token.")
else:
    # ---------------------------------------------------------------------
    # ESTRATÉGIA DE PROTEÇÃO: Definição das Estações Alvo
    # ---------------------------------------------------------------------
    estacoes_alvo = pd.DataFrame(columns=["codestacao", "estacao"])
    
    # 1. Tenta pegar do status mensal anterior
    if "df_status_maio_2026" in globals() and isinstance(df_status_maio_2026, pd.DataFrame) and not df_status_maio_2026.empty:
        # Força os nomes de coluna para minúsculo para evitar erros de digitação (case-sensitive)
        df_status_maio_2026.columns = df_status_maio_2026.columns.str.lower()
        if "status" in df_status_maio_2026.columns and "codestacao" in df_status_maio_2026.columns:
            # Filtra online ou pega todas se nenhuma estiver marcada como online
            df_filtrado = df_status_maio_2026[df_status_maio_2026["status"].astype(str).str.upper() == "ONLINE"]
            if df_filtrado.empty:
                df_filtrado = df_status_maio_2026
            
            est_col = "estacao" if "estacao" in df_filtrado.columns else "nome" if "nome" in df_filtrado.columns else "codestacao"
            estacoes_alvo = df_filtrado[["codestacao", est_col]].drop_duplicates().rename(columns={est_col: "estacao"}).copy()
            print("🔄 Estações carregadas a partir de 'df_status_maio_2026'.")

    # 2. Se falhou, tenta pegar do cadastro geotécnico geral df_geo
    if estacoes_alvo.empty and "df_geo" in globals() and isinstance(df_geo, pd.DataFrame) and not df_geo.empty:
        df_geo.columns = df_geo.columns.str.lower()
        if "codestacao" in df_geo.columns:
            est_col = "nome" if "nome" in df_geo.columns else "estacao" if "estacao" in df_geo.columns else "codestacao"
            estacoes_alvo = df_geo[["codestacao", est_col]].drop_duplicates().rename(columns={est_col: "estacao"}).copy()
            print("🔄 Estações carregadas a partir de 'df_geo'.")

    # 3. TRAVA DE SEGURANÇA (Fallback): Se tudo acima falhar ou estiver vazio, injeta as principais PCDs Geotécnicas de Recife
    if estacoes_alvo.empty:
        print("⚠️ Tabelas anteriores não encontradas ou vazias na memória. Ativando Banco de Dados interno de Recife...")
        dados_pcds_recife = {
            "codestacao": ["PE_01", "PE_02", "PE_03", "PE_04", "PE_05"],  # Substitua pelos códigos reais do seu projeto caso conheça
            "estacao": ["Dois Irmãos", "Campina do Barreto", "Imbiribeira", "Torreão", "Córrego do Jenipapo"]
        }
        estacoes_alvo = pd.DataFrame(dados_pcds_recife)

    # Executa a consulta se tivermos estações configuradas
    if estacoes_alvo.empty:
        print("Erro crítico: Nenhuma estacao alvo configurada.")
    else:
        # Configuração de Maio completo trancada para a API
        inicio = "202605010000"
        fim = "202605312359"

        blocos = []
        falhas = []

        print(f"🔎 Preparando consulta para {len(estacoes_alvo)} estações...")

        for _, est in estacoes_alvo.iterrows():
            params = {
                "codigo": str(est["codestacao"]).strip(),
                "inicio": inicio,
                "fim": fim,
                "rede": "11",
            }
            try:
                resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)
            except Exception as e:
                falhas.append({"codestacao": est["codestacao"], "estacao": est["estacao"], "status_http": f"Erro Conexao: {e}"})
                continue

            if resp.status_code == 401:
                token, err_tk = renovar_token_local()
                if not err_tk:
                    resp = requests.get(URL_CSV, params=params, headers={"token": token}, timeout=90)

            if not (resp.status_code == 200 and "cod.estacao" in resp.text):
                falhas.append({"codestacao": est["codestacao"], "estacao": est["estacao"], "status_http": resp.status_code})
                continue

            linhas = [ln for ln in resp.text.splitlines() if ln.strip()]
            if linhas and linhas[0].startswith("OBS."):
                linhas = linhas[1:]

            if len(linhas) <= 1:
                continue

            df_tmp = pd.read_csv(StringIO("\n".join(linhas)), sep=";")
            if df_tmp.empty:
                continue

            # Garante colunas em minúsculo no arquivo retornado para evitar incompatibilidades
            df_tmp.columns = df_tmp.columns.str.lower()

            # Converte UTC para hora local Recife
            col_data = "datahora" if "datahora" in df_tmp.columns else "data_hora" if "data_hora" in df_tmp.columns else None
            if col_data:
                dt_utc = pd.to_datetime(df_tmp[col_data], errors="coerce", utc=True)
                df_tmp["datahora_recife"] = dt_utc.dt.tz_convert("America/Recife").dt.tz_localize(None)

            # Filtro estrito de Maio
            if "datahora_recife" in df_tmp.columns:
                ini_local = pd.Timestamp("2026-05-01 00:00:00")
                fim_local = pd.Timestamp("2026-05-31 23:59:59")
                df_tmp = df_tmp[(df_tmp["datahora_recife"] >= ini_local) & (df_tmp["datahora_recife"] <= fim_local)].copy()

            df_tmp["estacao_nome"] = est["estacao"]
            df_tmp["codestacao_ref"] = est["codestacao"]
            blocos.append(df_tmp)

        # ---------------------------------------------------------------------
        # PROCESSAMENTO DOS BLOCOS RETORNADOS
        # ---------------------------------------------------------------------
        if not blocos:
            print("\n❌ NENHUM DADO RETORNADO. Motivos possíveis:\n"
                  "1. Os códigos das estações no fallback ('PE_01', etc.) precisam ser os códigos numéricos reais do Cemaden.\n"
                  "2. As estações selecionadas estão offline no servidor do Cemaden para este período.")
            if falhas:
                print("\nTabela de Falhas HTTP:")
                display(pd.DataFrame(falhas))
        else:
            df_maio_estacoes = pd.concat(blocos, ignore_index=True)

            # Padronização de nomes de colunas para saída acadêmica
            if "cod.estacao" in df_maio_estacoes.columns:
                df_maio_estacoes = df_maio_estacoes.rename(columns={"cod.estacao": "data_codestacao"})

            cols_inicio = [
                "codestacao_ref", "estacao_nome", "datahora_recife", "datahora",
                "sensor", "valor", "qualificacao", "municipio", "uf"
            ]
            cols_inicio = [c for c in cols_inicio if c in df_maio_estacoes.columns]
            df_maio_estacoes = df_maio_estacoes[cols_inicio + [c for c in df_maio_estacoes.columns if c not in cols_inicio]]

            if "datahora_recife" in df_maio_estacoes.columns:
                df_maio_estacoes = df_maio_estacoes.sort_values(["estacao_nome", "datahora_recife", "sensor"], na_position="last")

            print("\n=======================================================")
            print("🎉 DADOS DE MAIO/2026 EXTRAÍDOS COM SUCESSO!")
            print(f"   Estações com dados: {df_maio_estacoes['estacao_nome'].nunique()}")
            print(f"   Total de registros: {len(df_maio_estacoes)}")
            print("=======================================================\n")

            resumo = (
                df_maio_estacoes.groupby(["estacao_nome", "codestacao_ref"], dropna=False)
                .agg(
                    registros=("estacao_nome", "size"),
                    sensores=("sensor", lambda s: pd.Series(s).nunique()),
                    inicio_recife=("datahora_recife", "min"),
                    fim_recife=("datahora_recife", "max"),
                )
                .reset_index()
                .sort_values("registros", ascending=False)
            )
            display(resumo)
            display(df_maio_estacoes.head(100))

            nome_saida = "dados_maio_2026_estacoes_geotecnicas.csv"
            df_maio_estacoes.to_csv(nome_saida, index=False, encoding="utf-8-sig")
            print(f"\n💾 Arquivo salvo com sucesso: '{nome_saida}'")

            if falhas:
                print("\n⚠️ Detalhes das estações que falharam na requisição:")
                display(pd.DataFrame(falhas))

⚠️ Tabelas anteriores não encontradas ou vazias na memória. Ativando Banco de Dados interno de Recife...
🔎 Preparando consulta para 5 estações...

❌ NENHUM DADO RETORNADO. Motivos possíveis:
1. Os códigos das estações no fallback ('PE_01', etc.) precisam ser os códigos numéricos reais do Cemaden.
2. As estações selecionadas estão offline no servidor do Cemaden para este período.

Tabela de Falhas HTTP:


,codestacao,estacao,status_http
0,PE_01,Dois Irmãos,400
1,PE_02,Campina do Barreto,400
2,PE_03,Imbiribeira,400
3,PE_04,Torreão,400
4,PE_05,Córrego do Jenipapo,400
